## Combine, Clean & Status
Scans `RESULTS/` across all AL types and sizes, reports failures/missing,
combines successful results into `COMBINED/`, applies quality filters,
and produces per-job stats + per-size and overall MLPreprocessing visualizations.

In [ ]:
import sys, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from joblib import Parallel, delayed

THERMOIFT_SRC = Path("../../thermoift/src").resolve()
if str(THERMOIFT_SRC) not in sys.path:
    sys.path.insert(0, str(THERMOIFT_SRC))

from thermoift import MLPreprocessing

In [ ]:
RESULTS_DIR  = Path("RESULTS")
COMBINED_DIR = Path("COMBINED")
IFT_SUBPATH  = "CSV/InterfacialProperties/feed_1_interfacial_results.csv"

AL_TYPES = ["AL_ST", "AL_MT"]
SIZES    = ["N025", "N050", "N075", "N100"]

# Number of parallel workers — reads SLURM_CPUS_PER_TASK when running on a cluster
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

# Quality filters (same as PRE_ML)
MAX_VAPOR_DENSITY  = 400    # kg/m3 — above this is unphysical
MIN_IFT_THICKNESS  = 0.0    # nm   — negative thickness is unphysical
MAX_IFT_THICKNESS  = 7.5    # nm   — above this is unphysical / near-critical artifact
MIN_GAMMA          = 0.05   # mN/m — near-zero IFT is unphysical (near-critical artifact)

COMPONENTS = [
    "carbon dioxide", "hydrogen", "argon", "nitrogen",
    "methane", "oxygen", "carbon monoxide", "hydrogen sulfide"
]
TARGET = "gamma"
Z_COLS = [f"z_{c}" for c in COMPONENTS]

print(f"N_JOBS = {N_JOBS}")

In [ ]:
def run_ml_plots(df, out_folder, label=""):
    """Run all 7 MLPreprocessing plots into out_folder."""
    out_folder = Path(out_folder).resolve()
    out_folder.mkdir(parents=True, exist_ok=True)

    z_cols   = [c for c in Z_COLS if c in df.columns]
    features = ["temperature", "pressure"] + z_cols
    prep     = MLPreprocessing(df=df, features=features, target=TARGET)

    folder_str = str(out_folder)

    prep.plot_scatter("temperature", TARGET, "pressure",
                      save_path="gamma_vs_T",         folder=folder_str)
    plt.close("all")
    prep.plot_scatter("pressure",    TARGET, "temperature",
                      save_path="gamma_vs_P",         folder=folder_str)
    plt.close("all")
    prep.plot_scatter("liquid_density",        TARGET, "temperature",
                      save_path="gamma_vs_rhoL",      folder=folder_str)
    plt.close("all")
    prep.plot_scatter("vapor_density",         TARGET, "temperature",
                      save_path="gamma_vs_rhoV",      folder=folder_str)
    plt.close("all")
    prep.plot_scatter("interfacial_thickness",  TARGET, "temperature",
                      save_path="gamma_vs_thickness", folder=folder_str)
    plt.close("all")

    from thermoift import PLOT_SETTINGS as ps

    fig, ax = prep.plot_histogram(TARGET, bins=50, save_path=None)
    ps.save_plot(fig, "gamma_distribution", folder=folder_str)
    plt.close("all")

    fig, ax = prep.plot_phase_envelope(group_by="temperature", value_col=TARGET, save_path=None)
    ps.save_plot(fig, "gamma_phase_envelope", folder=folder_str)
    plt.close("all")

    print(f"  Plots → {out_folder}")

In [ ]:
def _process_job(al_type, size, task_map, combined_dir, ift_subpath,
                 min_ift, max_ift, min_gamma, max_rhov):
    """Read, filter, and write one (al_type, size) task group. Returns stats dict or None."""
    import pandas as pd
    from pathlib import Path

    out_dir  = Path(combined_dir) / al_type
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{size}.csv"

    dfs = []
    for task_id, folder in sorted(task_map.items()):
        try:
            df = pd.read_csv(Path(folder) / ift_subpath)
            df.insert(0, "task_id", task_id)
            dfs.append(df)
        except Exception as e:
            print(f"  Warning: {Path(folder).name}: {e}", flush=True)

    if not dfs:
        return None

    raw = pd.concat(dfs, ignore_index=True)

    mask_nan        = raw["gamma"].isna() | raw["interfacial_thickness"].isna()
    mask_thick_low  = raw["interfacial_thickness"] <= min_ift
    mask_thick_high = raw["interfacial_thickness"] > max_ift
    mask_gamma_low  = raw["gamma"] < min_gamma
    mask_rhoV       = raw["vapor_density"] >= max_rhov

    valid = ~mask_nan
    n_nan        = int(mask_nan.sum())
    n_thick_low  = int((valid & mask_thick_low).sum())
    n_thick_high = int((valid & ~mask_thick_low & mask_thick_high).sum())
    n_gamma_low  = int((valid & ~mask_thick_low & ~mask_thick_high & mask_gamma_low).sum())
    n_rhoV       = int((valid & ~mask_thick_low & ~mask_thick_high & ~mask_gamma_low & mask_rhoV).sum())
    n_raw        = len(raw)

    clean = raw[valid & ~mask_thick_low & ~mask_thick_high & ~mask_gamma_low & ~mask_rhoV]
    n_clean = len(clean)
    clean.to_csv(out_path, index=False)

    return {
        "al_type":       al_type,
        "size":          size,
        "tasks":         len(task_map),
        "raw_rows":      n_raw,
        "nan_gamma":     n_nan,
        "bad_thick_low": n_thick_low,
        "bad_thick_hi":  n_thick_high,
        "bad_gamma_low": n_gamma_low,
        "bad_rhoV":      n_rhoV,
        "clean_rows":    n_clean,
        "pct_clean":     round(100 * n_clean / n_raw, 1) if n_raw else 0.0,
    }


def _plot_one_al(al_type, size, combined_dir_str, thermoift_src_str, z_cols, target):
    """Generate all 7 MLPreprocessing plots for one (al_type, size). Runs in a loky worker."""
    import sys, matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import pandas as pd
    from pathlib import Path

    thermoift_src = Path(thermoift_src_str)
    if str(thermoift_src) not in sys.path:
        sys.path.insert(0, str(thermoift_src))
    from thermoift import MLPreprocessing, PLOT_SETTINGS as ps

    combined_dir = Path(combined_dir_str)
    csv = combined_dir / al_type / f"{size}.csv"
    if not csv.exists():
        return f"{al_type}/{size}: no combined CSV — skipping"

    df = pd.read_csv(csv)
    out_folder = (combined_dir / al_type / f"{size}_PLOTS").resolve()
    out_folder.mkdir(parents=True, exist_ok=True)
    folder_str = str(out_folder)

    z = [c for c in z_cols if c in df.columns]
    prep = MLPreprocessing(df=df, features=["temperature", "pressure"] + z, target=target)

    prep.plot_scatter("temperature", target, "pressure",           save_path="gamma_vs_T",         folder=folder_str); plt.close("all")
    prep.plot_scatter("pressure",    target, "temperature",        save_path="gamma_vs_P",         folder=folder_str); plt.close("all")
    prep.plot_scatter("liquid_density",        target, "temperature", save_path="gamma_vs_rhoL",   folder=folder_str); plt.close("all")
    prep.plot_scatter("vapor_density",         target, "temperature", save_path="gamma_vs_rhoV",   folder=folder_str); plt.close("all")
    prep.plot_scatter("interfacial_thickness",  target, "temperature", save_path="gamma_vs_thickness", folder=folder_str); plt.close("all")

    fig, ax = prep.plot_histogram(target, bins=50, save_path=None)
    ps.save_plot(fig, "gamma_distribution",   folder=folder_str); plt.close("all")

    fig, ax = prep.plot_phase_envelope(group_by="temperature", value_col=target, save_path=None)
    ps.save_plot(fig, "gamma_phase_envelope", folder=folder_str); plt.close("all")

    return f"{al_type}/{size}: {len(df):,} clean rows"

### Discover expected AL types and sizes from input CSVs

In [ ]:
expected = {}   # (al_type, size) -> n_compositions

for al_type in AL_TYPES:
    for size in SIZES:
        csv = Path(al_type) / "AL" / f"AL_{size}.csv"
        if csv.exists():
            n = sum(1 for _ in csv.open()) - 1
            expected[(al_type, size)] = n

print(f"Expected: {len(expected)} jobs")
for al_type in AL_TYPES:
    for size in SIZES:
        if (al_type, size) in expected:
            print(f"  {al_type}/{size}: {expected[(al_type, size)]} compositions")
        else:
            print(f"  {al_type}/{size}: CSV not found — skipped")

### Scan RESULTS/ and classify tasks

In [ ]:
failures  = {}
missing   = {}
successes = {}

for (al_type, size), n_expected in sorted(expected.items()):
    results_dir = RESULTS_DIR / al_type / size

    if not results_dir.exists():
        missing[(al_type, size)] = list(range(n_expected))
        continue

    by_task = defaultdict(list)
    for folder in results_dir.iterdir():
        if not folder.is_dir():
            continue
        parts = folder.name.rsplit("_", 1)
        if len(parts) == 2 and parts[1].isdigit():
            by_task[int(parts[1])].append(folder)

    task_failures  = []
    task_missing   = []
    task_successes = {}

    for task_id in range(n_expected):
        folders = by_task.get(task_id, [])
        if not folders:
            task_missing.append(task_id)
        else:
            ok = [f for f in folders if (f / IFT_SUBPATH).exists()]
            if not ok:
                task_failures.append(task_id)
            else:
                best = sorted(ok, key=lambda f: int(f.name.rsplit("_", 1)[0]))[-1]
                task_successes[task_id] = best

    if task_failures:
        failures[(al_type, size)] = task_failures
    if task_missing:
        missing[(al_type, size)] = task_missing
    if task_successes:
        successes[(al_type, size)] = task_successes

n_success = sum(len(v) for v in successes.values())
n_fail    = sum(len(v) for v in failures.values())
n_miss    = sum(len(v) for v in missing.values())
print(f"Successful tasks : {n_success}")
print(f"Failed tasks     : {n_fail}")
print(f"Missing tasks    : {n_miss}")

### What is missing / failed

In [ ]:
if not failures and not missing:
    print("All tasks completed successfully — nothing missing.")
else:
    if missing:
        print("=== Never ran ===")
        for (al_type, size), task_ids in sorted(missing.items()):
            if len(task_ids) == expected[(al_type, size)]:
                print(f"  {al_type}/{size}  →  entire job not submitted yet")
            else:
                print(f"  {al_type}/{size}  →  {len(task_ids)} tasks never ran: {task_ids}")

    if failures:
        print("\n=== Ran but failed ===")
        for (al_type, size), task_ids in sorted(failures.items()):
            print(f"  {al_type}/{size}  →  {len(task_ids)} failed tasks: {task_ids}")

    print("\n--- Resubmit with: ---")
    all_rerun = {**failures, **{k: v for k, v in missing.items() if len(v) < expected[k]}}
    for (al_type, size), task_ids in sorted(all_rerun.items()):
        ids = ",".join(str(i) for i in sorted(task_ids))
        print(f"  sbatch --array={ids}  RUN_AL.sh {al_type} {size}")

### Combine and clean — with per-job statistics

In [ ]:
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Combining {len(successes)} job groups with {N_JOBS} parallel workers...")

results = Parallel(n_jobs=N_JOBS, backend="multiprocessing")(
    delayed(_process_job)(
        al_type, size, task_map,
        COMBINED_DIR, IFT_SUBPATH,
        MIN_IFT_THICKNESS, MAX_IFT_THICKNESS, MIN_GAMMA, MAX_VAPOR_DENSITY
    )
    for (al_type, size), task_map in sorted(successes.items())
)

job_stats = [r for r in results if r is not None]
stats_df = pd.DataFrame(job_stats).sort_values(["al_type", "size"]).reset_index(drop=True)
print(f"Written {len(job_stats)} combined CSVs to {COMBINED_DIR}/")

### Per-job cleaning statistics

In [ ]:
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 120)
print(stats_df.to_string(index=False))

### Summary per AL type

In [ ]:
type_summary = (
    stats_df
    .groupby("al_type", sort=True)
    .agg(
        jobs          = ("size",          "count"),
        raw_rows      = ("raw_rows",      "sum"),
        nan_gamma     = ("nan_gamma",     "sum"),
        bad_thick_low = ("bad_thick_low", "sum"),
        bad_thick_hi  = ("bad_thick_hi",  "sum"),
        bad_gamma_low = ("bad_gamma_low", "sum"),
        bad_rhoV      = ("bad_rhoV",      "sum"),
        clean_rows    = ("clean_rows",    "sum"),
    )
)
type_summary["pct_clean"] = (100 * type_summary["clean_rows"] / type_summary["raw_rows"]).round(1)
print(type_summary.to_string())
print()

tot = stats_df[['raw_rows','nan_gamma','bad_thick_low','bad_thick_hi','bad_gamma_low','bad_rhoV','clean_rows']].sum()
print(f"TOTAL")
print(f"  Raw rows                          : {tot['raw_rows']:>8,}")
print(f"  Dropped NaN gamma                 : {tot['nan_gamma']:>8,}")
print(f"  Dropped thickness ≤ 0             : {tot['bad_thick_low']:>8,}")
print(f"  Dropped thickness > {MAX_IFT_THICKNESS} nm        : {tot['bad_thick_hi']:>8,}")
print(f"  Dropped gamma < {MIN_GAMMA} mN/m (near-crit): {tot['bad_gamma_low']:>8,}")
print(f"  Dropped rhoV ≥ {MAX_VAPOR_DENSITY} kg/m3          : {tot['bad_rhoV']:>8,}")
print(f"  Clean rows retained               : {tot['clean_rows']:>8,}  ({100*tot['clean_rows']/max(tot['raw_rows'],1):.2f}%)")

### Task completion summary

In [ ]:
print(f"{'AL Type':<10} {'Size':<8} {'N expected':>12} {'Tasks OK':>10} {'Failed':>8} {'Missing':>9}")
print("-" * 62)

for al_type in AL_TYPES:
    for size in SIZES:
        key = (al_type, size)
        if key not in expected:
            continue
        print(f"{al_type:<10} {size:<8} {expected[key]:>12} "
              f"{len(successes.get(key,{})):>10} "
              f"{len(failures.get(key,[])):>8} "
              f"{len(missing.get(key,[])):>9}")

### MLPreprocessing plots — per (AL type, size)
Saved to `COMBINED/<AL_TYPE>/<SIZE>_PLOTS/`.

In [ ]:
jobs_to_plot = [
    (al_type, size)
    for al_type in AL_TYPES
    for size in SIZES
    if (COMBINED_DIR / al_type / f"{size}.csv").exists()
]
print(f"Generating plots for {len(jobs_to_plot)} (AL type, size) combinations "
      f"with {min(N_JOBS, len(jobs_to_plot))} workers...")

msgs = Parallel(n_jobs=min(N_JOBS, len(jobs_to_plot)), backend="multiprocessing")(
    delayed(_plot_one_al)(al_type, size, str(COMBINED_DIR), str(THERMOIFT_SRC), Z_COLS, TARGET)
    for al_type, size in jobs_to_plot
)
for m in msgs:
    print(m)

### MLPreprocessing plots — overall (all AL types and sizes)
Saved to `COMBINED/PLOTS/`.

In [ ]:
all_dfs = []
for al_type in AL_TYPES:
    for size in SIZES:
        csv = COMBINED_DIR / al_type / f"{size}.csv"
        if csv.exists():
            df = pd.read_csv(csv)
            df.insert(0, "al_type", al_type)
            df.insert(1, "size", size)
            all_dfs.append(df)

if not all_dfs:
    print("No combined CSVs found.")
else:
    full = pd.concat(all_dfs, ignore_index=True)
    print(f"Full clean dataset: {len(full):,} rows")
    print(f"AL types: {full['al_type'].value_counts().to_dict()}")
    print(f"Sizes:    {full['size'].value_counts().sort_index().to_dict()}")
    print()
    print(full[["temperature", "pressure", "gamma", "interfacial_thickness",
                "liquid_density", "vapor_density"]].describe().round(3))
    print()
    run_ml_plots(full, COMBINED_DIR / "PLOTS", label="ALL")